# DICOM Data Organization and Sequence Swapping

This notebook is designed to automate the organization of PET-CT imaging data for medical research. Its primary functions are:

1.  **Drive Integration**: Connects to Google Drive to access shared imaging databases.
2.  **Sequence Alignment**: Iterates through patient folders to find specific study sequences that require re-ordering.
3.  **Automatic Swapping**:
    - It identifies **Whole Body (WB)** and **One Bed (OB)** sequences.
    - It swaps specific reconstructed folders between the `WB/OB` directories and the `PT` (standard PET) directory (e.g., swapping `NAC` reconstructions with `QC` ones).
4.  **Archiving**: After the necessary subfolders are swapped and reorganized, the script renames the source folders (`CT`, `PT`, `OT`) to include an `_archive` suffix to keep the directory structure clean while preserving original data.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [20]:
!cd "/content/drive/Shareddrives/Sandler_and_Kaminka/Ordered_DICOM/Bladder 13.11.25"

In [27]:
import os
import shutil
from google.colab import drive

# נתיב הבסיס לתיקייה
base_dir = "/content/drive/Shareddrives/Sandler_and_Kaminka/Ordered_DICOM/Bladder 13.11.25"

# ==========================================
# הגדרת כמות מטופלים לבדיקה (טסט)
# ==========================================
MAX_PATIENTS = None # שנה ל-None להרצה מלאה

patients_processed = 0

# מעבר על כל תיקיות המטופלים בתיקיית הבסיס
for patient_id in os.listdir(base_dir):

    patient_path = os.path.join(base_dir, patient_id)
    if not os.path.isdir(patient_path):
        continue

    # הגבלת כמות המטופלים לטסט
    if MAX_PATIENTS is not None and patients_processed >= MAX_PATIENTS:
        print(f"\n⏹️ Stopped after processing {MAX_PATIENTS} patients (Test Mode).")
        break

    print(f"\n➤ Processing patient: {patient_id}")
    patients_processed += 1

    for study_id in os.listdir(patient_path):
        if not study_id.startswith("Study_"):
            continue

        study_path = os.path.join(patient_path, study_id)
        if not os.path.isdir(study_path):
            continue

        # בדיקת אידמפוטנטיות - האם כבר עברנו כאן?
        # if os.path.exists(os.path.join(study_path, "PT_archive")):
        #     print(f"  ⏭️ Skipping {study_id} - Already processed.")
        #     continue

        pt_dir = os.path.join(study_path, "PT")
        wb_dir = os.path.join(study_path, "WB")
        ob_dir = os.path.join(study_path, "OB")

        # פונקציית עזר לביצוע החלפה בין שתי תיקיות על בסיס קידומת
        def swap_subfolders(dir_a, prefix_a, dir_b, prefix_b):
            if not (os.path.exists(dir_a) and os.path.exists(dir_b)):
                return False

            sub_a = next((d for d in os.listdir(dir_a) if d.startswith(prefix_a)), None)
            sub_b = next((d for d in os.listdir(dir_b) if d.startswith(prefix_b)), None)

            if sub_a and sub_b:
                print(f"  🔄 Swapping {sub_a} <-> {sub_b}")
                shutil.move(os.path.join(dir_a, sub_a), os.path.join(dir_b, sub_a))
                shutil.move(os.path.join(dir_b, sub_b), os.path.join(dir_a, sub_b))
                return True
            return False

        # ==========================================
        # שלב 1: החלפת מיקומים
        # ==========================================

        # החלפה 1: WB ו-PT (NAC מול QC)
        swapped_wb = swap_subfolders(wb_dir, "PT_WB_NAC_S", pt_dir, "PT_WB_QC_350_S")

        # החלפה 2: OB ו-PT (ONE_BED_HD מול ONE_BED_QC)
        swapped_ob = swap_subfolders(ob_dir, "PT_OB_ONE_BED_HD_S", pt_dir, "PT_ONE_BED_QC_S")

        if not swapped_wb and not swapped_ob:
            print(f"  ⚠️ No matching subfolder pairs found for swap in {study_id}")

        # ==========================================
        # שלב 2: הוספת סיומת archive
        # ==========================================
        for folder_name in ["CT", "PT", "OT"]:
            old_path = os.path.join(study_path, folder_name)
            new_path = os.path.join(study_path, f"{folder_name}_archive")
            if os.path.exists(old_path):
                print(f"  📁 Archiving {folder_name}...")
                os.rename(old_path, new_path)

print("\n✅ Finished script execution!")


➤ Processing patient: 31311736
  ⚠️ No matching subfolder pairs found for swap in Study_1.2.826.0.1.3680043.8.498.62064878589153395807303313701326198788

➤ Processing patient: 31020402
  ⚠️ No matching subfolder pairs found for swap in Study_1.2.826.0.1.3680043.8.498.87428959254225525824397723785448839500

➤ Processing patient: 31207009
  ⚠️ No matching subfolder pairs found for swap in Study_1.2.826.0.1.3680043.8.498.50098091635218090960449668521922052736

➤ Processing patient: 31164138
  ⚠️ No matching subfolder pairs found for swap in Study_1.2.826.0.1.3680043.8.498.10798356029224303427632700454337819245

➤ Processing patient: 31299410
  🔄 Swapping PT_WB_NAC_S311 <-> PT_WB_QC_350_S311
  🔄 Swapping PT_OB_ONE_BED_HD_S79 <-> PT_ONE_BED_QC_S79
  📁 Archiving CT...
  📁 Archiving PT...
  📁 Archiving OT...

➤ Processing patient: 31399172
  🔄 Swapping PT_WB_NAC_S311 <-> PT_WB_QC_350_S311
  🔄 Swapping PT_OB_ONE_BED_HD_S79 <-> PT_ONE_BED_QC_S79
  📁 Archiving CT...
  📁 Archiving PT...
  📁 Arch

In [28]:
print("Nun of processed", patients_processed)

Nun of processed 66
